# TripPulse — Week 4: Source-to-Bronze Ingestion

**Notebook:** `notebooks/02_bronze_ingestion.ipynb`  
**Project:** TripPulse — Urban Mobility Analytics  
**Week 4 scope:** Approved batch source files → persistent Bronze Delta tables

## Objective

Move the approved TripPulse batch source files into persistent Bronze Delta tables while preserving source business values as received and adding ingestion/lineage metadata.

This notebook does **not** perform Silver cleaning, deduplication, business transformations, Gold aggregation, Power BI work, or streaming ingestion.


## 1. Environment setup

TripPulse Unity Catalog Volume:

`/Volumes/trippulse/default/trippulsedata`

This notebook uses catalog `TripPulse` and the existing `default` schema. It does not create a new schema.


In [ ]:
%sql
USE CATALOG `TripPulse`;
USE SCHEMA `default`;

SELECT
  current_catalog() AS active_catalog,
  current_schema() AS active_schema;


## 2. Source inventory

| Source file | Format | Bronze table |
|---|---|---|
| `zones.csv` | CSV | `bronze_trippulse_zones` |
| `drivers.json` | JSON array | `bronze_trippulse_drivers` |
| `trips.parquet` | Parquet | `bronze_trippulse_trips` |
| `payments.csv` | CSV | `bronze_trippulse_payments` |

Only controlled batch sources are included in Week 4. Streaming ride-request event/drop files are reserved for later weeks.


## 3. Confirm source files are visible


In [ ]:
%fs ls /Volumes/trippulse/default/trippulsedata


## 4. Controlled ingestion run

A single run ID is generated once and reused by all four Bronze loads in this notebook execution.


In [ ]:
from uuid import uuid4

INGESTION_RUN_ID = str(uuid4())
SCHEMA_VERSION = "week04_v1"

print("Ingestion run ID:", INGESTION_RUN_ID)
print("Schema version:", SCHEMA_VERSION)


# 5. Source 1 — `zones.csv`

**Format:** CSV  
**Path:** `/Volumes/trippulse/default/trippulsedata/zones.csv`  
**Target:** `bronze_trippulse_zones`

The Bronze layer preserves the source values and adds technical metadata.


In [ ]:
zones_path = "/Volumes/trippulse/default/trippulsedata/zones.csv"

zones_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(zones_path)
)

zones_df.createOrReplaceTempView("zones_source")

display(zones_df.limit(10))
zones_df.printSchema()
print("zones source count =", zones_df.count())


In [ ]:
from pyspark.sql import functions as F

zones_business_cols = zones_df.columns

zones_bronze_df = (
    zones_df
    .withColumn("_source_file_name", F.lit("zones.csv"))
    .withColumn("_source_file_path", F.lit(zones_path))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumn("_schema_version", F.lit(SCHEMA_VERSION))
    .withColumn(
        "_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[F.coalesce(F.col(c).cast("string"), F.lit("<NULL>")) for c in zones_business_cols]
            ),
            256
        )
    )
)

zones_bronze_df.createOrReplaceTempView("zones_bronze_ready")
display(zones_bronze_df.limit(10))


In [ ]:
%sql
CREATE OR REPLACE TABLE bronze_trippulse_zones
USING DELTA
AS
SELECT * FROM zones_bronze_ready;


In [ ]:
%sql
SELECT * FROM bronze_trippulse_zones LIMIT 10;


In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM zones_source) AS source_count,
  (SELECT COUNT(*) FROM bronze_trippulse_zones) AS bronze_count,
  CASE
    WHEN (SELECT COUNT(*) FROM zones_source) =
         (SELECT COUNT(*) FROM bronze_trippulse_zones)
    THEN 'MATCH'
    ELSE 'CHECK'
  END AS reconciliation_status;


# 6. Source 2 — `drivers.json`

**Format:** pretty-printed JSON array  
**Path:** `/Volumes/trippulse/default/trippulsedata/drivers.json`  
**Target:** `bronze_trippulse_drivers`

The Week 3 exploration established that this file is a JSON array, so `multiLine=True` is required.


In [ ]:
drivers_path = "/Volumes/trippulse/default/trippulsedata/drivers.json"

drivers_df = (
    spark.read
    .option("multiLine", True)
    .json(drivers_path)
)

drivers_df.createOrReplaceTempView("drivers_source")

display(drivers_df.limit(10))
drivers_df.printSchema()
print("drivers source count =", drivers_df.count())


In [ ]:
drivers_business_cols = drivers_df.columns

drivers_bronze_df = (
    drivers_df
    .withColumn("_source_file_name", F.lit("drivers.json"))
    .withColumn("_source_file_path", F.lit(drivers_path))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumn("_schema_version", F.lit(SCHEMA_VERSION))
    .withColumn(
        "_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[F.coalesce(F.col(c).cast("string"), F.lit("<NULL>")) for c in drivers_business_cols]
            ),
            256
        )
    )
)

drivers_bronze_df.createOrReplaceTempView("drivers_bronze_ready")
display(drivers_bronze_df.limit(10))


In [ ]:
%sql
CREATE OR REPLACE TABLE bronze_trippulse_drivers
USING DELTA
AS
SELECT * FROM drivers_bronze_ready;


In [ ]:
%sql
SELECT * FROM bronze_trippulse_drivers LIMIT 10;


In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM drivers_source) AS source_count,
  (SELECT COUNT(*) FROM bronze_trippulse_drivers) AS bronze_count,
  CASE
    WHEN (SELECT COUNT(*) FROM drivers_source) =
         (SELECT COUNT(*) FROM bronze_trippulse_drivers)
    THEN 'MATCH'
    ELSE 'CHECK'
  END AS reconciliation_status;


# 7. Source 3 — `trips.parquet`

**Format:** Parquet  
**Path:** `/Volumes/trippulse/default/trippulsedata/trips.parquet`  
**Target:** `bronze_trippulse_trips`

## Important TripPulse reader requirement

Week 3 identified a project-specific issue: six timestamp fields in this Parquet file are physically stored as Parquet `INT64 TIMESTAMP(NANOS)`. Current Databricks/Spark cannot infer that physical type directly.

For Bronze, we preserve those source values without converting them into cleaned timestamps. Therefore, the explicit Week 3 schema reads those six fields as `LongType`.


In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, LongType
)

trips_path = "/Volumes/trippulse/default/trippulsedata/trips.parquet"

trips_schema = StructType([
    StructField("trip_id",               StringType(), True),
    StructField("request_ts",            LongType(),   True),
    StructField("driver_accept_ts",      LongType(),   True),
    StructField("pickup_ts",             LongType(),   True),
    StructField("dropoff_ts",            LongType(),   True),
    StructField("cancel_ts",             LongType(),   True),
    StructField("driver_id",             StringType(), True),
    StructField("pickup_zone_id",        StringType(), True),
    StructField("dropoff_zone_id",       StringType(), True),
    StructField("service_type",          StringType(), True),
    StructField("trip_status",           StringType(), True),
    StructField("cancellation_reason",   StringType(), True),
    StructField("estimated_distance_km", DoubleType(), True),
    StructField("actual_distance_km",    DoubleType(), True),
    StructField("estimated_fare_inr",    DoubleType(), True),
    StructField("final_fare_inr",        DoubleType(), True),
    StructField("surge_multiplier",      DoubleType(), True),
    StructField("record_created_ts",     LongType(),   True),
])

trips_df = spark.read.schema(trips_schema).parquet(trips_path)
trips_df.createOrReplaceTempView("trips_source")

display(trips_df.limit(10))
trips_df.printSchema()
print("trips source count =", trips_df.count())


In [ ]:
trips_business_cols = trips_df.columns

trips_bronze_df = (
    trips_df
    .withColumn("_source_file_name", F.lit("trips.parquet"))
    .withColumn("_source_file_path", F.lit(trips_path))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumn("_schema_version", F.lit(SCHEMA_VERSION))
    .withColumn(
        "_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[F.coalesce(F.col(c).cast("string"), F.lit("<NULL>")) for c in trips_business_cols]
            ),
            256
        )
    )
)

trips_bronze_df.createOrReplaceTempView("trips_bronze_ready")
display(trips_bronze_df.limit(10))


In [ ]:
%sql
CREATE OR REPLACE TABLE bronze_trippulse_trips
USING DELTA
AS
SELECT * FROM trips_bronze_ready;


In [ ]:
%sql
SELECT * FROM bronze_trippulse_trips LIMIT 10;


In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM trips_source) AS source_count,
  (SELECT COUNT(*) FROM bronze_trippulse_trips) AS bronze_count,
  CASE
    WHEN (SELECT COUNT(*) FROM trips_source) =
         (SELECT COUNT(*) FROM bronze_trippulse_trips)
    THEN 'MATCH'
    ELSE 'CHECK'
  END AS reconciliation_status;


# 8. Source 4 — `payments.csv`

**Format:** CSV  
**Path:** `/Volumes/trippulse/default/trippulsedata/payments.csv`  
**Target:** `bronze_trippulse_payments`

One source row represents a payment attempt. Bronze keeps that source grain unchanged.


In [ ]:
payments_path = "/Volumes/trippulse/default/trippulsedata/payments.csv"

payments_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(payments_path)
)

payments_df.createOrReplaceTempView("payments_source")

display(payments_df.limit(10))
payments_df.printSchema()
print("payments source count =", payments_df.count())


In [ ]:
payments_business_cols = payments_df.columns

payments_bronze_df = (
    payments_df
    .withColumn("_source_file_name", F.lit("payments.csv"))
    .withColumn("_source_file_path", F.lit(payments_path))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumn("_schema_version", F.lit(SCHEMA_VERSION))
    .withColumn(
        "_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[F.coalesce(F.col(c).cast("string"), F.lit("<NULL>")) for c in payments_business_cols]
            ),
            256
        )
    )
)

payments_bronze_df.createOrReplaceTempView("payments_bronze_ready")
display(payments_bronze_df.limit(10))


In [ ]:
%sql
CREATE OR REPLACE TABLE bronze_trippulse_payments
USING DELTA
AS
SELECT * FROM payments_bronze_ready;


In [ ]:
%sql
SELECT * FROM bronze_trippulse_payments LIMIT 10;


In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM payments_source) AS source_count,
  (SELECT COUNT(*) FROM bronze_trippulse_payments) AS bronze_count,
  CASE
    WHEN (SELECT COUNT(*) FROM payments_source) =
         (SELECT COUNT(*) FROM bronze_trippulse_payments)
    THEN 'MATCH'
    ELSE 'CHECK'
  END AS reconciliation_status;


# 9. Consolidated reconciliation


In [ ]:
%sql
SELECT
  'zones.csv' AS source_file,
  (SELECT COUNT(*) FROM zones_source) AS source_count,
  (SELECT COUNT(*) FROM bronze_trippulse_zones) AS bronze_count,
  CASE WHEN (SELECT COUNT(*) FROM zones_source) =
            (SELECT COUNT(*) FROM bronze_trippulse_zones)
       THEN 'MATCH' ELSE 'CHECK' END AS status

UNION ALL

SELECT
  'drivers.json',
  (SELECT COUNT(*) FROM drivers_source),
  (SELECT COUNT(*) FROM bronze_trippulse_drivers),
  CASE WHEN (SELECT COUNT(*) FROM drivers_source) =
            (SELECT COUNT(*) FROM bronze_trippulse_drivers)
       THEN 'MATCH' ELSE 'CHECK' END

UNION ALL

SELECT
  'trips.parquet',
  (SELECT COUNT(*) FROM trips_source),
  (SELECT COUNT(*) FROM bronze_trippulse_trips),
  CASE WHEN (SELECT COUNT(*) FROM trips_source) =
            (SELECT COUNT(*) FROM bronze_trippulse_trips)
       THEN 'MATCH' ELSE 'CHECK' END

UNION ALL

SELECT
  'payments.csv',
  (SELECT COUNT(*) FROM payments_source),
  (SELECT COUNT(*) FROM bronze_trippulse_payments),
  CASE WHEN (SELECT COUNT(*) FROM payments_source) =
            (SELECT COUNT(*) FROM bronze_trippulse_payments)
       THEN 'MATCH' ELSE 'CHECK' END;


# 10. Verify Bronze metadata

Every Bronze table should contain the source business columns plus:

- `_source_file_name`
- `_source_file_path`
- `_ingested_at`
- `_ingestion_run_id`
- `_schema_version`
- `_record_hash`


In [ ]:
%sql
SELECT
  '_source_file_name' AS metadata_column,
  SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END) AS null_count
FROM bronze_trippulse_trips
UNION ALL
SELECT '_source_file_path', SUM(CASE WHEN _source_file_path IS NULL THEN 1 ELSE 0 END)
FROM bronze_trippulse_trips
UNION ALL
SELECT '_ingested_at', SUM(CASE WHEN _ingested_at IS NULL THEN 1 ELSE 0 END)
FROM bronze_trippulse_trips
UNION ALL
SELECT '_ingestion_run_id', SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END)
FROM bronze_trippulse_trips
UNION ALL
SELECT '_schema_version', SUM(CASE WHEN _schema_version IS NULL THEN 1 ELSE 0 END)
FROM bronze_trippulse_trips
UNION ALL
SELECT '_record_hash', SUM(CASE WHEN _record_hash IS NULL THEN 1 ELSE 0 END)
FROM bronze_trippulse_trips;


# 11. Controlled repeat-run test

Use `trips.parquet` as the rerun-tested source.

Record the Bronze count **before** the rerun, execute the same `CREATE OR REPLACE TABLE` write again, then record the count **after**. Because this is a controlled replacement load, the row count should not double.


In [ ]:
%sql
SELECT COUNT(*) AS bronze_count_before_rerun
FROM bronze_trippulse_trips;


In [ ]:
%sql
CREATE OR REPLACE TABLE bronze_trippulse_trips
USING DELTA
AS
SELECT * FROM trips_bronze_ready;


In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM trips_source) AS expected_source_count,
  COUNT(*) AS bronze_count_after_rerun,
  CASE
    WHEN COUNT(*) = (SELECT COUNT(*) FROM trips_source)
    THEN 'PASS - NO UNINTENDED DUPLICATION'
    ELSE 'CHECK'
  END AS rerun_status
FROM bronze_trippulse_trips;


# 12. Delta details and history


In [ ]:
%sql
DESCRIBE DETAIL bronze_trippulse_trips;


In [ ]:
%sql
DESCRIBE HISTORY bronze_trippulse_trips;


# 13. Confirm all TripPulse Bronze tables


In [ ]:
%sql
SHOW TABLES LIKE 'bronze_trippulse_*';


# 14. Week 4 completion checklist

Before closing Week 4, confirm:

- all four approved batch files are visible in the Volume;
- all four sources are readable;
- one persistent Bronze Delta table exists per source;
- source business values were preserved without Silver transformations;
- technical ingestion metadata is present;
- source and Bronze counts reconcile;
- the controlled `trips` rerun did not unintentionally duplicate rows;
- Delta history was inspected;
- all Bronze tables are visible in Catalog Explorer;
- screenshots/evidence are saved under `evidence/week_04/`;
- `weekly/week_04_log.md` is updated;
- the notebook is committed as `notebooks/02_bronze_ingestion.ipynb`;
- the AI Transparency Note records where AI was used and what the team manually verified.

## Week 4 boundary

Do not add Silver cleaning, Gold KPIs, Power BI, or streaming implementation to this notebook.
